# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Abdullah-Nazeer/Flyrank-Ml-Pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **One row means:** one page (content item) on one day for one client, with its Search Console numbers (`report_date` × `client_hash_id` × `content_hash_id`).
2. **Table:** `fact_content_daily_performance`, month=2026-03 only (GSC columns).
3. **Time window:** features from 2026-03-01 to 2026-03-15; label from 2026-03-16 to 2026-03-31. The final month (2026-06) is sealed and not touched.
4. **Predict:** whether a page gets at least one click in days 16-31 (label = future_clicks above the median, which is 0 here), using only days 1-15.
5. **Deliberately excluded:** all GA4 columns (a different lane) and every column from days 16-31 as a feature, since that is the outcome window.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| Feature | past_impressions, past_clicks, past_avg_position, past_ctr, past_days_with_data | all computed from days 1-15 only |
| Label | future_clicks (turned into above/below median) | outcome in days 16-31 |
| Context | client_hash_id, content_hash_id, report_date | identify the row and split by client, never fed to the model |
| Excluded | all ga4_*, sessions_*, ai_* columns; any gsc column dated after 03-15 as a feature | other lane / would leak the future |

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
GRAIN = "report_date, client_hash_id, content_hash_id"
print("grain:", GRAIN)

grain: report_date, client_hash_id, content_hash_id


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
CREATE OR REPLACE TABLE d AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print(con.sql("SELECT COUNT(*) AS rows_in_month FROM d").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_in_month
0        9841378


In [13]:
# Query 1: grain
display(con.sql(f"""
SELECT COUNT(*) AS n_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT {GRAIN} FROM d)) AS distinct_keys
FROM d""").df())

# Query 2: row count and date span of my slice (clients with GSC)
display(con.sql("""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
FROM d WHERE client_has_gsc IS TRUE""").df())

# Query 3: availability with IS TRUE
display(con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving
FROM d""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,distinct_keys
0,9841378,9841378


,n_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


,total_rows,rows_surviving
0,9841378,3611061


In [14]:
feat = con.sql("""
WITH past AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS past_impressions,
         SUM(gsc_clicks)      AS past_clicks,
         AVG(gsc_avg_position) AS past_avg_position,
         COUNT(*)             AS past_days_with_data
  FROM d
  WHERE gsc_data_available IS TRUE AND report_date <= DATE '2026-03-15'
  GROUP BY 1, 2),
fut AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS future_clicks
  FROM d
  WHERE gsc_data_available IS TRUE AND report_date > DATE '2026-03-15'
  GROUP BY 1, 2)
SELECT p.*, p.past_clicks * 1.0 / NULLIF(p.past_impressions, 0) AS past_ctr, f.future_clicks
FROM past p JOIN fut f USING (client_hash_id, content_hash_id)
""").df()

feat["label"] = (feat["future_clicks"] > feat["future_clicks"].median()).astype(int)
FEATURES = ["past_impressions", "past_clicks", "past_avg_position", "past_ctr", "past_days_with_data"]
print(feat.shape, feat["label"].mean())
display(feat[FEATURES + ["label"]].describe().T)

(141467, 9) 0.35953968063223224


,count,mean,std,min,25%,50%,75%,max
past_impressions,141467.0,900.674765,2763.944947,1.0,23.000000,134.000000,678.000000,161575.0
past_clicks,141467.0,2.717581,13.705023,0.0,0.000000,0.000000,1.000000,2395.0
past_avg_position,141467.0,15.767910,17.394127,0.0,4.850308,8.461343,20.224664,310.0
past_ctr,141467.0,0.003979,0.029368,0.0,0.000000,0.000000,0.002284,1.0
past_days_with_data,141467.0,11.410909,4.760916,1.0,8.000000,14.000000,15.000000,15.0
label,141467.0,0.359540,0.479867,0.0,0.000000,0.000000,1.000000,1.0


**What the queries showed:**
- Grain: 9,841,378 rows and 9,841,378 distinct (report_date, client_hash_id, content_hash_id) keys, so one row really is one page on one day for one client. No duplicates.
- Slice: 9,841,378 rows spanning 2026-03-01 to 2026-03-31, the full month. Every row belongs to a client with Search Console, so the client_has_gsc filter removed nothing.
- Availability: with `gsc_data_available IS TRUE`, 3,611,061 of 9,841,378 rows survive (about 36.7%). The other 63.3% have no Search Console data that day and are excluded.
- After building features from days 1-15 and the label from days 16-31, 141,467 pages (client × content item) have data in both halves.

- `past_impressions`: knowable at the decision moment because it only sums 03-01 to 03-15, which has already happened.
- `past_clicks`: knowable at the decision moment because it only counts clicks up to 03-15.
- `past_avg_position`: knowable at the decision moment because it averages the daily ranking position up to 03-15.
- `past_ctr`: knowable at the decision moment because it is past clicks divided by past impressions, both from before 03-16.
- `past_days_with_data`: knowable at the decision moment because it counts how many days up to 03-15 had Search Console data.

In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    tr, te = next(GroupShuffleSplit(test_size=0.25, random_state=0)
                  .split(feat, groups=feat["client_hash_id"]))
    m = HistGradientBoostingClassifier(random_state=0).fit(feat.iloc[tr][cols], feat.iloc[tr]["label"])
    return roc_auc_score(feat.iloc[te]["label"], m.predict_proba(feat.iloc[te][cols])[:, 1])

honest = quick_score(FEATURES)

feat["leak"] = feat["future_clicks"]            # THE TRAP: made from the outcome window
leaky = quick_score(FEATURES + ["leak"])
print(f"honest AUC = {honest:.3f}   with leak = {leaky:.3f}")

feat = feat.drop(columns=["leak"])              # delete it, keep the honest number
print(f"honest AUC (kept) = {quick_score(FEATURES):.3f}")

honest AUC = 0.832   with leak = 1.000
honest AUC (kept) = 0.832


**Leak experiment:** honest AUC = 0.832; with the deliberately leaked `future_clicks` column it jumped to 1.000, so I deleted it and keep 0.832 as the honest number.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation of my slice:** only one month (March 2026) and only clients with Search Console data, so it cannot show seasonality, and per-client history depth differs (an unbalanced panel). Pages with no GSC data in either half of the month are dropped by the join, so the result only describes pages that appeared in both halves.
The label is imbalanced (about 36% positive) and most pages have zero clicks, so the task is really "will this page get any clicks", not "will it be a top performer".

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Numbers behind the limitation (March 2026 slice)
print("pages in both halves:", len(feat))
print("share with label = 1 (got clicks in days 16-31):", round(feat["label"].mean(), 3))
print("median future_clicks:", feat["future_clicks"].median())
print("share of pages with zero future clicks:", round((feat["future_clicks"] == 0).mean(), 3))
print("share of rows with GSC data available:", round(3611061 / 9841378, 3))

pages in both halves: 141467
share with label = 1 (got clicks in days 16-31): 0.36
median future_clicks: 0.0
share of pages with zero future clicks: 0.64
share of rows with GSC data available: 0.367


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.